In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
os.chdir("..")

In [4]:
import torch
import json
import numpy as np
from transformers import AutoTokenizer
from tqdm.auto import tqdm
from datasets import load_dataset
from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation, THINK_TOKEN

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

compute_dtype = torch.float
device   = 'cuda'
model_id = "Qwen/QwQ-32B"

In [5]:
from pathlib import Path

cur_dir = Path(".").absolute()


def load_dataset_from_file(domain_name, task_name):
    prompt_dir = cur_dir / Path(f"./cot-planning/results/{domain_name}/qwq-32b/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)

In [7]:
task_name = "plan_generation_po"
eval_results = [
    load_dataset_from_file(domain_name, task_name)["instances"] for domain_name in [
        "blocksworld_mystery",
    ]
]
eval_results = [{x["dataset_idx"]: x for x in er} for er in eval_results]

In [8]:

tokenizer = initialize_tokenizer(model_id)

In [9]:
dataset = load_dataset(f"dmitriihook/qwq-32b-planning-mystery-24k")["train"]

In [ ]:
DOMAIN_PHRASES = {
    "mystery_1": {
        "actions": {
            "attack": "attack",
            "succumb": "succumb",
            "overcome": "overcome",
            "feast": "feast"
        },
        "predicates": {
            "planet": "planet",
            "province": "province",
            "harmony": "harmony",
            "craves": "craves",
            "pain": "pain"
        }
    },
    "mystery_2": {
        "actions": {
            "attack": "illuminate",
            "succumb": "silence",
            "overcome": "distill",
            "feast": "divest"
        },
        "predicates": {
            "planet": "aura",
            "province": "essence",
            "harmony": "nexus",
            "craves": "harmonizes",
            "pain": "pulse"
        }
    },
}

In [ ]:
def extract_all_phrase_positions(tokens, phrase, tokenizer, cot_only=True, min_pos=None):
    """Find end of the phrase token positions"""
    tokens = tokens.squeeze()

    phrase_tokens = [
        tokenizer.encode(" " + phrase),
        tokenizer.encode(" " + phrase.capitalize()),
        tokenizer.encode("\n" + phrase)[1:],
        tokenizer.encode("\n" + phrase.capitalize())[1:],
        tokenizer.encode("\n\n" + phrase)[1:],
        tokenizer.encode("\n\n" + phrase.capitalize())[1:],
    ]

    positions = set()

    if cot_only:
        start_pos = torch.where(tokens == 151667)[0]
        start_mask = torch.arange(tokens.shape[0]) >= start_pos

    for phts in phrase_tokens:
        presence_mask = torch.ones_like(tokens)
        if cot_only:
            presence_mask = presence_mask * start_mask

        for i, t in enumerate(phts):
            presence_mask = presence_mask * (tokens == t)[i:]
            presence_mask = presence_mask[:-1]

        for p in (torch.where(presence_mask)[0]).tolist():
            if min_pos is not None and p < min_pos:
                continue
            positions.add(
                tuple([p-1, p + len(phts)])
            )        
    
    return sorted(list(set(positions)))

: 

In [10]:
from vllm import LLM

llm = LLM(model=model_id, tensor_parallel_size=4, enforce_eager=True, max_seq_len_to_capture=20000, max_num_batched_tokens=4096)

INFO 04-11 18:00:56 __init__.py:190] Automatically detected platform cuda.
INFO 04-11 18:01:09 config.py:542] This model supports multiple tasks: {'score', 'generate', 'classify', 'embed', 'reward'}. Defaulting to 'generate'.
INFO 04-11 18:01:09 config.py:1401] Defaulting to use mp for distributed inference
WARNING 04-11 18:01:09 arg_utils.py:1135] Chunked prefill is enabled by default for models with max_model_len > 32K. Currently, chunked prefill might not work with some features or models. If you encounter any issues, please disable chunked prefill by setting --enable-chunked-prefill=False.
INFO 04-11 18:01:09 config.py:1556] Chunked prefill is enabled with max_num_batched_tokens=4096.
WARNING 04-11 18:01:09 cuda.py:95] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
WARNING 04-11 18:01:09 config.py:678] Async output processing is not supported on the current platform type cuda.
INFO 04-11 18:01:09

/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


(VllmWorkerProcess pid=499070) [2025-04-11 18:01:15,188] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)
(VllmWorkerProcess pid=499065) [2025-04-11 18:01:15,225] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)
(VllmWorkerProcess pid=499062) [2025-04-11 18:01:15,239] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)
INFO 04-11 18:01:15 cuda.py:230] Using Flash Attention backend.


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


(VllmWorkerProcess pid=499070) INFO 04-11 18:01:15 cuda.py:230] Using Flash Attention backend.
(VllmWorkerProcess pid=499062) INFO 04-11 18:01:15 cuda.py:230] Using Flash Attention backend.
(VllmWorkerProcess pid=499065) INFO 04-11 18:01:15 cuda.py:230] Using Flash Attention backend.
INFO 04-11 18:01:17 utils.py:950] Found nccl from library libnccl.so.2
INFO 04-11 18:01:17 pynccl.py:69] vLLM is using nccl==2.21.5
(VllmWorkerProcess pid=499065) (VllmWorkerProcess pid=499062) (VllmWorkerProcess pid=499070) INFO 04-11 18:01:17 utils.py:950] Found nccl from library libnccl.so.2
INFO 04-11 18:01:17 utils.py:950] Found nccl from library libnccl.so.2
INFO 04-11 18:01:17 utils.py:950] Found nccl from library libnccl.so.2
(VllmWorkerProcess pid=499065) (VllmWorkerProcess pid=499062) (VllmWorkerProcess pid=499070) INFO 04-11 18:01:17 pynccl.py:69] vLLM is using nccl==2.21.5
INFO 04-11 18:01:17 pynccl.py:69] vLLM is using nccl==2.21.5
INFO 04-11 18:01:17 pynccl.py:69] vLLM is using nccl==2.21.5
I

Loading safetensors checkpoint shards:   0% Completed | 0/14 [00:00<?, ?it/s]


(VllmWorkerProcess pid=499070) INFO 04-11 18:01:21 weight_utils.py:252] Using model weights format ['*.safetensors']
(VllmWorkerProcess pid=499062) INFO 04-11 18:01:21 weight_utils.py:252] Using model weights format ['*.safetensors']
INFO 04-11 18:01:27 model_runner.py:1115] Loading model weights took 15.3937 GB
(VllmWorkerProcess pid=499065) INFO 04-11 18:01:27 model_runner.py:1115] Loading model weights took 15.3937 GB
(VllmWorkerProcess pid=499070) INFO 04-11 18:01:27 model_runner.py:1115] Loading model weights took 15.3937 GB
(VllmWorkerProcess pid=499062) INFO 04-11 18:01:28 model_runner.py:1115] Loading model weights took 15.3937 GB
(VllmWorkerProcess pid=499062) INFO 04-11 18:01:31 worker.py:267] Memory profiling takes 3.51 seconds
(VllmWorkerProcess pid=499062) INFO 04-11 18:01:31 worker.py:267] the current vLLM instance can use total_gpu_memory (79.10GiB) x gpu_memory_utilization (0.90) = 71.19GiB
(VllmWorkerProcess pid=499062) INFO 04-11 18:01:31 worker.py:267] model weights 

In [11]:
from pathlib import Path

cur_dir = Path(".").absolute()

def load_dataset_from_file(domain_name, task_name, model_name):
    prompt_dir = cur_dir / Path(f"./cot-planning/results/{domain_name}/{model_name}/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)["instances"]
    
eval_results = [
    load_dataset_from_file("blocksworld_mystery_2", "plan_generation_po", "qwq-32b-original-full-1-il-60-greedy"),
]

In [12]:
print(dataset[6]["generation"])

Okay, let's tackle this problem step by step. First, I need to understand the initial conditions and the goal clearly. 

The initial conditions are:
- Aura Block B
- Essence Block D
- Block A harmonizes Block C
- Block C harmonizes Block B
- Block D harmonizes Block A
- Nexus is present.

The goal is to have:
- Block A harmonizes Block B
- Block B harmonizes Block C
- Block D harmonizes Block A (which is already true, so that's already satisfied).

Wait, actually, the third part of the goal is "Block D harmonizes Block A", which is already in the initial conditions. So we just need to achieve the first two: A harmonizes B and B harmonizes C. 

Looking at the existing harmonies, we have A-C and C-B. So A-C-B forms a chain. But we need A-B and B-C. Wait, B-C is already there? Wait, the initial conditions say "Block C harmonizes Block B", which is the same as B harmonizes C (since harmonizing is probably mutual?), but maybe not. Wait, the problem might treat harmonizes as a directed relat

In [13]:
eval_results[0][99]["llm_correct"]

True

In [14]:
all_tokens = []

for i in range(100):
    row = dataset[i]

    text = "\n\n".join(row["generation"].split("\n\n")[:60])
    tokens = tokenize_blocksworld_generation(tokenizer, row, text)[:, :-2][0]
    
    all_tokens.append(tokens)

In [15]:
tokens = all_tokens[0]

In [16]:
len(tokens)

1967

In [17]:
with open(
    "mean_reprs_mystery_2.json",
    'r'
) as f:
    reprs = json.load(f)

In [18]:
with open(
    "mystery_representations_greedy/mystery_2/mean_reprs_mystery_2.json"
) as f:
    reprs_greedy = json.load(f)

In [19]:
mean_reprs = {
    k: np.array(v) for k, v in reprs["mean_reprs"].items()
}

In [20]:
mean_actions = np.array(reprs["mean_domain"])
mean_predicates = np.array(reprs["mean_domain"])

In [21]:
phrases = DOMAIN_PHRASES["mystery_2"]
phrases = list(phrases["actions"].values()) + list(phrases["predicates"].values())

In [22]:
phrase_positions = {
    phrase: extract_all_phrase_positions(tokens, phrase, tokenizer, cot_only=False, min_pos=0)
    for phrase in phrases
}

In [23]:
pharse_masks = {
    phrase: np.zeros(tokens.shape[0])
    for phrase in phrases
}

for ip, phrase in enumerate(phrases):
    positions = phrase_positions[phrase]
    
    for start, end in positions:
        pharse_masks[phrase][start:end] = 1
        
    pharse_masks


In [24]:
np.where(pharse_masks[phrases[0]])

(array([  20,   21,   55,   56,   76,   77,   92,   93,  402,  403,  404,
         415,  416,  417,  554,  555,  556,  567,  568,  569,  580,  581,
         582,  827,  828,  859,  860, 1388, 1389, 1393, 1394, 1427, 1428,
        1431, 1432, 1535, 1536, 1672, 1673, 1843, 1844]),)

In [25]:
tokenizer.decode(tokens[pharse_masks["distill"] == 1])

'   Distill perform Distill Once Distill Once Distill\ndistill\ndistill\ndistill\ndistill\ndistill, Distill **Distill the Distill To Distill to DistillMaybe Distill can Distill, Distill to Distill in Distill "Distill is Distill to Distill to Distill after Distill this Distill: DistillAfter Distill'

In [26]:

masks_batch = {
    k: [v, v, v, v] for k, v in pharse_masks.items()
}

masks_batch_combined = {
    k: np.concatenate(v, axis=0) for k, v in masks_batch.items()
}

combined_len = masks_batch_combined[phrases[0]].shape[0]

In [27]:
masks_batch_combined["illuminate"].shape

(7868,)

In [28]:
mask_start = 0
mask_end = 4096

for ip, phrase in enumerate(phrases[:1]):
    if ip < 4:
        adjustment = mean_actions
    else:
        adjustment = mean_predicates
    
    steering_mask = masks_batch_combined[phrase]
    
    steering_mask = steering_mask[mask_start:mask_end]
    # steering_mask = np.concatenate([steering_mask, np.zeros(hs.shape[0] - steering_mask.shape[0])], axis=0)
    
    steering_vector = mean_reprs[phrase] - adjustment
    steering_vector = steering_mask[:, None] * steering_vector

In [29]:
mean_reprs[phrases[0]] - mean_predicates

array([-0.07421875, -0.42456055, -0.81860352, ...,  3.40625   ,
       -0.25939941,  0.21142578])

In [30]:

from collections import OrderedDict

block_size = 4096

def hook(module, input, output):
    meta = getattr(module, "_meta", {})
    meta["mask_offset"] = meta.get("mask_offset", 0)
    hs, res = output        
    
    if meta["mask_offset"] >= combined_len:
        return output
    
    mask_start = meta["mask_offset"]
    mask_end = mask_start + block_size
    
    meta["mask_offset"] = mask_end
    module._meta = meta
    
    
    for ip, phrase in enumerate(phrases):
        if ip < 4:
            adjustment = mean_actions
        else:
            adjustment = mean_predicates
        
        steering_mask = masks_batch_combined[phrase]
        
        steering_mask = steering_mask[mask_start:mask_end]
        steering_mask = np.concatenate([steering_mask, np.zeros(hs.shape[0] - steering_mask.shape[0])], axis=0)
        
        steering_vector = mean_reprs[phrase] - adjustment
        steering_vector = steering_mask[:, None] * steering_vector
        steering_vector = torch.tensor(steering_vector, dtype=hs.dtype, device=hs.device)
        
        steering_mask = torch.tensor(steering_mask[:, None], dtype=hs.dtype, device=hs.device)
        
        if ip == 0:
            torch.save(
                steering_vector.cpu(),
                f"steering_vector_{mask_start}_{mask_end}.pt"
            )
    
        # hs += steering_vector * 1
        # res += steering_vector * 1
        
        hs = (1 - steering_mask) * hs + steering_mask * (hs * 0.5 + steering_vector * 0.5)
    torch.save(
        {
            "hs": hs.cpu(),
            "res": res.cpu()
        },
        f"vllm_hidden_states/hs_0_{mask_start}_{mask_end}.pt"
    )
    # hs[0][0] += 12
    
    return hs, res


# def hook(module, input, kwargs):
#     meta = getattr(module, "_meta", {})
#     meta["mask_offset"] = meta.get("mask_offset", 0)  
    
#     if meta["mask_offset"] >= combined_len:
#         return None
    
#     mask_start = meta["mask_offset"]
#     mask_end = mask_start + block_size
    
#     meta["mask_offset"] = mask_end
#     module._meta = meta
    
#     hs = input[1]
    
#     for ip, phrase in enumerate(phrases):
#         if ip < 4:
#             adjustment = mean_actions
#         else:
#             adjustment = mean_predicates
        
#         steering_mask = masks_batch_combined[phrase]
        
#         steering_mask = steering_mask[mask_start:mask_end]
#         steering_mask = np.concatenate([steering_mask, np.zeros(hs.shape[0] - steering_mask.shape[0])], axis=0)
        
#         steering_vector = mean_reprs[phrase] - adjustment
#         steering_vector = steering_mask[:, None] * steering_vector
#         steering_vector = torch.tensor(steering_vector, dtype=hs.dtype, device=hs.device)
        
#         if ip == 0:
#             torch.save(
#                 steering_vector.cpu(),
#                 f"steering_vector_{mask_start}_{mask_end}.pt"
#             )
    
#         # hs += steering_vector * 1
#         hs += steering_vector * 1
    
#     # input[1] = hs
    
#     return (input[0], hs) + input[2:], kwargs


def logging_hook(module, input, output):
    meta = getattr(module, "_meta", {})
    meta["mask_offset_2"] = meta.get("mask_offset_2", 0)
    
    if meta["mask_offset_2"] >= combined_len:
        return output
    
    mask_start = meta["mask_offset_2"]
    mask_end = mask_start + block_size
    
    meta["mask_offset_2"] = mask_end
    module._meta = meta
    
    hs, res = output
    
    torch.save(
        {
            "hs": hs.cpu(),
            "res": res.cpu()
        },
        f"vllm_hidden_states/0_{mask_start}_{mask_end}.pt"
    )

def logging_hook(module, input, kwargs):
    meta = getattr(module, "_meta", {})
    meta["mask_offset_2"] = meta.get("mask_offset_2", 0)
    
    if meta["mask_offset_2"] >= combined_len:
        return

    mask_start = meta["mask_offset_2"]
    mask_end = mask_start + block_size
    
    meta["mask_offset_2"] = mask_end
    module._meta = meta
    
    torch.save(
        {
            "hs": input[1].cpu(),
            "res": input[1].cpu()
        },
        f"vllm_hidden_states/0_{mask_start}_{mask_end}.pt"
    )
    
    

def add_hook(module, pre_hooks, hooks):
    module._forward_hooks = OrderedDict()
    module._forward_pre_hooks = OrderedDict()
    module._meta = {}
    for hook in hooks:
        module.register_forward_hook(hook)
    for hook in pre_hooks:
        module.register_forward_pre_hook(hook, with_kwargs=True)

    

In [50]:
def empty_hook(module, input, kwargs):
    pass


def f(x):
    # add_hook(x.model.layers[47], [], [hook])
    add_hook(x.model.layers[47], [], [hook])
    add_hook(x.model.layers[48], [], [])
    add_hook(x.model.layers[49], [logging_hook], [])

llm.apply_model(
   f,
)

[None, None, None, None]

In [51]:
from vllm import SamplingParams
sampling_params = SamplingParams(
    max_tokens=5,
    temperature=0,
    top_k=1,
)

In [52]:
from vllm import TokensPrompt

prompt = TokensPrompt(
    prompt_token_ids=tokens.tolist(),
)

In [53]:
res = llm.generate(
    [prompt, prompt, prompt, prompt], sampling_params=sampling_params
)

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 4/4 [00:02<00:00,  1.45it/s, est. speed input: 2853.07 toks/s, output: 7.25 toks/s]


In [54]:
dumped_hs = {}
dumped_res = {}

for path in Path("vllm_hidden_states").glob("*.pt"):
    with open(path, 'rb') as f:
        data = torch.load(f)
        dumped_hs[path.stem] = data["hs"]
        dumped_res[path.stem] = data["res"]

/tmp/ipykernel_495780/3549528137.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(f)


In [55]:
dumped_sv = {}
for path in Path(".").glob("steering_vector_*.pt"):
    with open(path, 'rb') as f:
        data = torch.load(f)
        dumped_sv[path.stem] = data

/tmp/ipykernel_495780/825171416.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(f)


In [56]:
dumped_sv['steering_vector_0_4096'][21]

tensor([-0.0742, -0.4238, -0.8203,  ...,  3.4062, -0.2598,  0.2109],
       dtype=torch.bfloat16)

In [57]:
hs_clean = dumped_hs["0_0_4096_clean"]
hs_clean_2 = hs_clean[len(tokens):len(tokens) + len(tokens)]

res_clean = dumped_res["0_0_4096_clean"]
res_clean_2 = res_clean[len(tokens):len(tokens) + len(tokens)]

hs = dumped_hs["0_0_4096"]
hs_2 = hs[len(tokens):len(tokens) + len(tokens)]

res = dumped_res["0_0_4096"]
res_2 = res[len(tokens):len(tokens) + len(tokens)]

res.shape

torch.Size([4096, 5120])

In [58]:
hs_clean = dumped_hs["0_0_4096_clean"]
hs_clean_1 = hs_clean[:len(tokens)]

res_clean = dumped_res["0_0_4096_clean"]
res_clean_1 = res_clean[:len(tokens)]

hs = dumped_hs["0_0_4096"]
hs_1 = hs[:len(tokens)]

res = dumped_res["0_0_4096"]
res_1 = res[:len(tokens)]

In [59]:
hs_hf_clean = torch.load("hf_hidden_states/0_clean_48.pt")
hs_hf = torch.load("hf_hidden_states/0_new_48.pt")

/tmp/ipykernel_495780/3142370261.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  hs_hf_clean = torch.load("hf_hidden_states/0_clean_48.pt")
/tmp/ipykernel_495780/3142370

In [ ]:
(dumped_hs["hs_0_0_4096"][:len(tokens)] - hs_clean_1)[:23]

tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [-0.1211,  0.0977,  0.0742,  ...,  1.8984,  0.4648,  0.5312],
        [ 0.1230,  0.6406, -0.7109,  ...,  1.1328, -0.5117,  0.2676],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
       dtype=torch.bfloat16)

: 

In [61]:
(res_1 - res_clean_1)[:23]

tensor([[-0.0156, -1.1641, -1.4609,  ..., -3.5000, -2.2344, -1.9062],
        [ 0.4902, -0.0254,  0.1006,  ..., -0.1895,  0.3164, -0.0654],
        [ 1.7031, -0.0703,  0.5703,  ..., -1.4844, -0.1094,  1.4297],
        ...,
        [-0.6914,  0.0703, -0.2812,  ...,  0.4551,  2.2344,  1.3359],
        [-0.3672,  0.2422, -0.2246,  ..., -1.5078,  0.2656, -0.1113],
        [ 1.3984,  2.4375,  0.2480,  ..., -2.7656,  0.5977, -0.7070]],
       dtype=torch.bfloat16)

In [62]:
(res_2 - res_clean_2)[:23]

tensor([[-0.0391, -1.1641, -1.4531,  ..., -3.5000, -2.2656, -1.8594],
        [ 0.4922, -0.0254,  0.1006,  ..., -0.1904,  0.3164, -0.0654],
        [ 1.7188, -0.0586,  0.5898,  ..., -1.4922, -0.1367,  1.4062],
        ...,
        [-0.6836,  0.0898, -0.1797,  ...,  0.4688,  2.1875,  1.3203],
        [-0.4199,  0.2266, -0.1543,  ..., -1.5391,  0.2695, -0.1465],
        [ 1.3750,  2.4062,  0.2344,  ..., -2.6250,  0.6016, -0.7070]],
       dtype=torch.bfloat16)

In [63]:
(hs_hf - hs_hf_clean)[:23]

tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [-0.3965,  0.0273, -0.9492,  ...,  1.3125,  1.4297,  1.2969],
        [ 2.2500, -0.5977, -0.7812,  ..., -3.9375, -2.5156, -1.6719],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
       dtype=torch.bfloat16)

In [131]:
dumped_sv['steering_vector_0_4096'][21]

tensor([-0.0742, -0.4238, -0.8203,  ...,  3.4062, -0.2598,  0.2109],
       dtype=torch.bfloat16)

In [152]:
np.linalg.norm((hs - hs_clean - (hs_hf - hs_hf_clean)).cpu().float().numpy())

2215.1377

In [110]:
(hs_hf - hs_hf_clean)[-100:]

tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [-0.5430, -0.2500, -0.1641,  ..., -1.2500,  0.2734,  0.3828],
        ...,
        [ 1.0703,  0.1328,  0.1016,  ..., -6.4375, -0.5781, -0.2031],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
       dtype=torch.bfloat16)

In [35]:
print(res[0].outputs[0].text)

 Let's see which actions can be performed now.

Check each action's prerequisites:

1. Illuminate any object: To do Illuminate on an object, say X, we need Essence X, Aura X, and Nexus. Let's check for each object:

- For Block A: Essence? No (initial has Essence B and D). Aura A is present. So no Essence A, can't Illuminate A.

- Block B: Essence B is present, Aura B is present. So can Illuminate B. That would create Pulse B, remove Essence B and Aura B.

- Block C: Essence? No. So can't Illuminate.

- Block D: Essence D is present, but Aura D? The initial conditions don't mention Aura D. The initial has aura Block A and B. So no Aura D. So can't Illuminate D.

So the only possible Illuminate is on B.

2. Silence any object: requires Pulse of the object. Currently, no objects have Pulse. So can't do Silence yet.

3. Distill: requires Essence of other object and Pulse of the object. Let's see possible Distill actions.

Suppose we want to Distill X from Y. Need Essence Y and Pulse X.

L

In [35]:
print(res[1].outputs[0].text)

 Let's list possible actions based on current facts.

Possible actions:

1. Illuminate any object that has Essence and Aura and Nexus. Let's check:

- For Block A: has Aura (yes), Essence? No (initial Essence is B and D). So can't Illuminate A yet.

- Block B: has Essence (yes), Aura (yes). So can Illuminate B. Doing so would create Pulse B, remove Essence B and Aura B.

- Block D: has Essence (yes), Aura? No (initial Aura is A and B). So can't Illuminate D.

- Block C: has Essence? No. So can't Illuminate.

So the only possible Illuminate is on B. Let's consider that.

If we Illuminate B:

- Precondition: Essence B (yes), Aura B (yes), Nexus (yes). So yes.

- Post: Pulse B becomes true. Essence B and Aura B become false.

So after Illuminate B:

- Pulse B is true.

- Essence B is gone, Aura B is gone.

- The other facts remain except those changed.

Now, with Pulse B, maybe we can do something else. For example, Silence B would require Pulse B, which is now present. Let's see:

Silenc

In [36]:
print(res[0].outputs[0].text)

 Let's list possible actions based on current facts.

Possible actions:

1. Illuminate any object that has Essence and Aura and Nexus. Let's check:

- For Block A: has Aura (yes), Essence? No (initial Essence is B and D). So can't Illuminate A yet.

- Block B: has Essence (yes), Aura (yes). So can Illuminate B. Doing so would create Pulse B, remove Essence B and Aura B.

- Block D: has Essence (yes), Aura? No (initial Aura is A and B). So can't Illuminate D.

- Block C: has Essence? No. So can't Illuminate.

So the only possible Illuminate is on B. Let's consider that.

If we Illuminate B:

- Precondition: Essence B (yes), Aura B (yes), Nexus (yes). So yes.

- Post: Pulse B becomes true. Essence B and Aura B become false.

So after Illuminate B:

- Pulse B is true.

- Essence B is gone, Aura B is gone.

- The other facts remain except those changed.

Now, with Pulse B, maybe we can do something else. For example, Silence B would require Pulse B, which is now present. Let's see:

Silenc

In [67]:
len(res[0].outputs[0].token_ids), len(res[1].outputs[0].token_ids)

(12881, 10259)

In [ ]:
print(res[0].outputs[0].text)

NameError: name 'res' is not defined

: 

In [31]:
res[1].outputs[0].text

' Let\'s see which actions can be performed now.\n\nCheck each action\'s prerequisites:\n\n1. Illuminate any object: To do Illuminate on an object, say X, we need Essence X, Aura X, and Nexus. Let\'s check for each object:\n\n- For Block A: Essence? No (initial Essence is B and D). Aura A is present. So can\'t do Illuminate A yet.\n\n- Block B: Essence B is present, Aura B is present. So can do Illuminate B. That would create Pulse B, remove Essence B and Aura B.\n\n- Block C: Essence? No. So can\'t.\n\n- Block D: Essence D is present, but Aura D? The initial conditions don\'t mention Aura D. The initial has aura Block A and B. So no Aura D. So can\'t Illuminate D.\n\nSo the only possible Illuminate now is on Block B.\n\n2. Silence any object: requires Pulse of the object. Currently, no objects have Pulse. So can\'t do Silence yet.\n\n3. Distill any object from another: requires Essence of the other object and Pulse of the object. Let\'s see:\n\nSuppose we want to Distill X from Y. Nee

In [46]:
print(res[1].outputs[0].text)

 Let's see which actions can be performed now.

Check each action's prerequisites:

1. Illuminate any object: To do Illuminate on an object, say X, we need Essence X, Aura X, and Nexus. Let's check for each object:

- For Block A: Essence? No (initial has Essence B and D). Aura A is present. So no Essence A, so can't Illuminate A.

- Block B: Essence B is present, Aura B is present. So can Illuminate B. That would create Pulse B, remove Essence B and Aura B.

- Block C: Essence? No. So can't Illuminate.

- Block D: Essence D is present, Aura? No (initial has Aura A and B). So can't Illuminate D.

So the only possible Illuminate is on B. Let's consider that.

If we Illuminate B:

- After Illuminate B:

   - Pulse B is true.

   - Essence B and Aura B become false.

   - So now, Essence B is gone, Aura B is gone, but Pulse B is on.

   Then, perhaps we can use that Pulse for some Distill action.

Alternatively, maybe we can do other actions first.

Silence action requires Pulse of an obj

In [30]:
599 + 4096

4695

In [31]:
for idx in range(100):
    row = dataset[idx]
    text = "\n\n".join(row["generation"].split("\n\n")[:40])
    tokens = tokenize_blocksworld_generation(tokenizer, row, text)[:, :-2][0]
    
    print(
        idx, len(tokens)
    )

0 1751
1 1994
2 1752
3 2347
4 1961
5 1884
6 1840
7 1857
8 1820
9 2229
10 1982
11 1997
12 1709
13 1984
14 1996
15 2209
16 1967
17 1815
18 2200
19 1973
20 1928
21 2094
22 2003
23 2189
24 1674
25 2037
26 2212
27 1641
28 2393
29 1962
30 1915
31 3441
32 2083
33 2047
34 2139
35 1769
36 1987
37 2539
38 2133
39 2130
40 1791
41 2055
42 1945
43 2076
44 2221
45 3061
46 2251
47 2372
48 1858
49 2470
50 2067
51 2376
52 2255
53 2002
54 1960
55 1886
56 2298
57 1691
58 2101
59 1759
60 2605
61 2101
62 2147
63 1802
64 2187
65 2141
66 1947
67 2028
68 2116
69 2163
70 2189
71 2706
72 2354
73 2199
74 2393
75 1844
76 2407
77 2426
78 2317
79 2379
80 1794
81 1776
82 2380
83 2029
84 1846
85 2441
86 2020
87 2153
88 2079
89 2171
90 2025
91 2014
92 2316
93 2428
94 2016
95 1789
96 2063
97 2357
98 2043
99 2516


In [32]:
print(res[0].outputs[0].text)

 Let's see which actions can be performed now.

Check each action's prerequisites:

1. Illuminate any object: To do Illuminate on an object, say X, we need Essence X, Aura X, and Nexus. Let's check for each object:

- For Block A: Essence? No (initial has Essence B and D). Aura A is present. So no Essence A, can't Illuminate A.

- Block B: Essence B is present, Aura B is present. So can Illuminate B. That would create Pulse B, remove Essence B and Aura B.

- Block C: Essence? No. So can't Illuminate.

- Block D: Essence D is present, but Aura D? The initial conditions don't mention Aura D. The initial has aura Block A and B. So no Aura D. So can't Illuminate D.

So the only possible Illuminate is on B.

2. Silence any object: requires Pulse of the object. Currently, no objects have Pulse. So can't do Silence yet.

3. Distill: requires Essence of other object and Pulse of the object. Let's see possible Distill actions.

Suppose we want to Distill X from Y. Need Essence Y and Pulse X.

L

In [33]:
print(
    row["generation"]
)

Okay, let's tackle this problem step by step. First, I need to understand the initial conditions and the goal clearly. 

The initial conditions are:
- Aura Block A
- Aura Block C
- Essence Block C
- Essence Block D
- Block B harmonizes Block A
- Block D harmonizes Block B
- Nexus

The goal is to have:
- Block A harmonizes Block B
- Block C harmonizes Block D
- Block D harmonizes Block A

Hmm, so the current harmonizes are B-A and D-B. The goal adds A-B (which is already there but maybe in reverse?), C-D, and D-A. Wait, actually, the goal requires A harmonizes B (which is already true as B harmonizes A? Or is harmonizes directional? The problem might treat it as a bidirectional relationship, but the actions might require specific directions. Wait, the problem says "Block B harmonizes Block A" is initial, and the goal is "Block A harmonizes Block B". So maybe they are separate? Or perhaps the harmonizes is symmetric? The problem might require both directions, but the initial has B harmon